# Standalone AA-CLIP Training on Kaggle

This notebook clones the official AA-CLIP repository, downloads the required OpenAI CLIP backbone, uses the Kaggle VisA/MVTec dataset inputs, trains AA-CLIP adapters with upstream `train.py`, and zips the resulting checkpoints.

Kaggle internet must be enabled for cloning AA-CLIP and downloading the OpenAI CLIP backbone. Add the Kaggle dataset input that exposes `/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922`.

In [ ]:
from pathlib import Path

WORKING_DIR = Path('/kaggle/working')
AACLIP_ROOT = WORKING_DIR / 'AA-CLIP'
AACLIP_REPO = 'https://github.com/Mwxinnn/AA-CLIP.git'
OPENCLIP_WEIGHT_URL = 'https://openaipublic.azureedge.net/clip/models/3035c92b350959924f9f00213499208652fc7ea050643e8b385c2dac08641f02/ViT-L-14-336px.pt'
OPENCLIP_WEIGHT_SHA256 = '3035c92b350959924f9f00213499208652fc7ea050643e8b385c2dac08641f02'

# Kaggle dataset input paths from the shared dataset collection.
MVTEC_PATH = Path('/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection')
VISA_PATH = Path('/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922')

# Official AA-CLIP training setup from scripts.sh: train on VisA with full_shot.
# Kaggle T4 usually cannot fit AA-CLIP's official batch sizes at img_size=518,
# so keep the setup and reduce only batch sizes for VRAM safety.
DATASET_NAME = 'VisA'
TRAINING_MODE = 'full_shot'  # 'full_shot' or 'few_shot'
SHOT = 32                  # only used when TRAINING_MODE='few_shot'
IMG_SIZE = 518
TEXT_BATCH_SIZE = 1
IMAGE_BATCH_SIZE = 1
TEXT_EPOCH = 5
IMAGE_EPOCH = 20
TEXT_LR = 1e-5
IMAGE_LR = 5e-4
SEED = 111

# Timing probe only: run one image-adapter epoch first, skipping text-adapter training.
# Use this to estimate image-stage runtime on Kaggle. Set back to False for final training.
IMAGE_STAGE_TIME_CHECK = False

run_name = f'{DATASET_NAME.lower()}_{TRAINING_MODE}_{SHOT}shot'
if IMAGE_STAGE_TIME_CHECK:
    run_name += '_image_time_check'
SAVE_PATH = WORKING_DIR / 'aaclip_ckpt' / run_name
SAVE_PATH.mkdir(parents=True, exist_ok=True)

print('AA-CLIP training config')
print('  repo:', AACLIP_REPO)
print('  dataset:', DATASET_NAME)
print('  mode:', TRAINING_MODE)
print('  image-stage timing check:', IMAGE_STAGE_TIME_CHECK)
print('  VisA path:', VISA_PATH)
print('  MVTec path:', MVTEC_PATH)
print('  output:', SAVE_PATH)


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

DEPS_MARKER = Path('/kaggle/working/.aaclip_deps_installed')

def dependency_check():
    code = """
import sys
import numpy as np
major = int(np.__version__.split('.')[0])
if major >= 2:
    raise SystemExit(f'NumPy must be <2 for this OpenCV wheel, got {np.__version__}')
import cv2
print('numpy:', np.__version__)
print('cv2:', cv2.__version__)
"""
    return subprocess.run([sys.executable, '-c', code], text=True, capture_output=True)

check = dependency_check()
if check.returncode != 0:
    print('Dependency check failed; reinstalling pinned NumPy/OpenCV stack.')
    print(check.stdout)
    print(check.stderr)
    %pip uninstall -y -q opencv-python opencv-python-headless numpy
    %pip install -q --no-cache-dir --force-reinstall "numpy==1.26.4" "opencv-python-headless==4.9.0.80"
    %pip install -q --no-cache-dir "numpy==1.26.4" einops==0.7.0 ftfy==6.2.0 kornia==0.6.9 timm==0.6.12 tiktoken==0.7.0 ipdb pandas scikit-learn scikit-image matplotlib tqdm
    DEPS_MARKER.write_text('installed')
    print('Installed compatible dependencies. Kaggle must restart the kernel now. Click OK, then run all cells from the top once more.')
    os.kill(os.getpid(), 9)
else:
    print('Dependencies verified in a fresh Python subprocess:')
    print(check.stdout)


In [ ]:
import hashlib
import json
import os
import random
import shutil
import subprocess
import sys
import urllib.request
from tqdm.auto import tqdm

def run(cmd, cwd=None, env=None):
    print('+', ' '.join(map(str, cmd)))
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True, env=merged_env)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def download_file(url, target, expected_sha256=None):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and (expected_sha256 is None or sha256(target) == expected_sha256):
        print('Already downloaded:', target)
        return target
    print('Downloading:', url)
    with urllib.request.urlopen(url) as response:
        total = int(response.headers.get('Content-Length', 0))
        with open(target, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as bar:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                f.write(chunk)
                bar.update(len(chunk))
    if expected_sha256 is not None:
        actual = sha256(target)
        if actual != expected_sha256:
            raise RuntimeError(f'SHA256 mismatch for {target}: expected {expected_sha256}, got {actual}')
    return target

def clone_aaclip():
    if (AACLIP_ROOT / 'train.py').exists():
        run(['git', 'pull', '--ff-only'], cwd=AACLIP_ROOT)
    else:
        run(['git', 'clone', AACLIP_REPO, AACLIP_ROOT])
    os.chdir(AACLIP_ROOT)
    if str(AACLIP_ROOT) not in sys.path:
        sys.path.insert(0, str(AACLIP_ROOT))

clone_aaclip()
print('AA-CLIP ready at:', AACLIP_ROOT)


In [ ]:
def prepare_openclip_weight():
    target = AACLIP_ROOT / 'model' / 'ViT-L-14-336px.pt'
    download_file(OPENCLIP_WEIGHT_URL, target, OPENCLIP_WEIGHT_SHA256)
    print('OpenAI CLIP backbone ready:', target)

prepare_openclip_weight()


In [ ]:
DATASET_PATHS = {
    'VisA': VISA_PATH,
    'MVTec': MVTEC_PATH,
}

def verify_dataset_root(dataset_name, dataset_root):
    dataset_root = Path(dataset_root)
    if not dataset_root.exists():
        raise FileNotFoundError(
            f'{dataset_name} path does not exist: {dataset_root}. '
            'Add the Kaggle dataset input or fix the path in the first cell.'
        )
    meta_file = AACLIP_ROOT / 'dataset' / 'metadata' / dataset_name / 'full-shot.jsonl'
    first = json.loads(next(line for line in meta_file.read_text().splitlines() if line.strip()))
    sample_image = dataset_root / first['image_path']
    if not sample_image.exists():
        raise FileNotFoundError(
            f'{dataset_name} metadata sample not found: {sample_image}. '
            'The dataset root must be the directory containing AA-CLIP JSONL-relative paths.'
        )
    print(f'Verified {dataset_name}:', dataset_root)
    print('  sample:', sample_image)
    return dataset_root.resolve()

def patch_dataset_constants():
    constants_path = AACLIP_ROOT / 'dataset' / 'constants.py'
    text = constants_path.read_text(encoding='utf-8')
    marker = '# Kaggle dataset path overrides'
    if marker in text:
        text = text.split(marker)[0].rstrip() + '\n'
    verified = {name: verify_dataset_root(name, path) for name, path in DATASET_PATHS.items()}
    lines = [f"DATA_PATH[{name!r}] = r'{path.as_posix()}'" for name, path in verified.items()]
    override = '\n' + marker + '\n' + '\n'.join(lines) + '\n'
    constants_path.write_text(text.rstrip() + override, encoding='utf-8')
    print('Patched AA-CLIP dataset paths.')

def ensure_few_shot_metadata():
    if TRAINING_MODE != 'few_shot':
        return
    meta_dir = AACLIP_ROOT / 'dataset' / 'metadata' / DATASET_NAME
    target = meta_dir / f'{SHOT}-shot.jsonl'
    if target.exists():
        print('Few-shot metadata already exists:', target)
        return
    rows = [json.loads(line) for line in (meta_dir / 'full-shot.jsonl').read_text().splitlines() if line.strip()]
    rng = random.Random(SEED)
    by_class = {}
    for row in rows:
        by_class.setdefault(row['class_name'], []).append(row)
    selected = []
    for _, class_rows in sorted(by_class.items()):
        class_rows = class_rows[:]
        rng.shuffle(class_rows)
        selected.extend(class_rows[:min(SHOT, len(class_rows))])
    with target.open('w', encoding='utf-8') as f:
        for row in selected:
            f.write(json.dumps(row) + '\n')
    print('Created:', target, 'rows:', len(selected))

patch_dataset_constants()
ensure_few_shot_metadata()


In [ ]:
import torch

# Official AA-CLIP train.py uses tqdm progress bars inside each epoch.
# The -u flag keeps subprocess output unbuffered so Kaggle displays progress live.
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

effective_text_epoch = 0 if IMAGE_STAGE_TIME_CHECK else TEXT_EPOCH
effective_image_epoch = 1 if IMAGE_STAGE_TIME_CHECK else IMAGE_EPOCH
if IMAGE_STAGE_TIME_CHECK:
    print('IMAGE_STAGE_TIME_CHECK=True: skipping text-adapter training and running one image-adapter epoch.')

train_cmd = [
    sys.executable, '-u', 'train.py',
    '--dataset', DATASET_NAME,
    '--training_mode', TRAINING_MODE,
    '--shot', str(SHOT),
    '--save_path', str(SAVE_PATH),
    '--img_size', str(IMG_SIZE),
    '--text_batch_size', str(TEXT_BATCH_SIZE),
    '--image_batch_size', str(IMAGE_BATCH_SIZE),
    '--text_epoch', str(effective_text_epoch),
    '--image_epoch', str(effective_image_epoch),
    '--text_lr', str(TEXT_LR),
    '--image_lr', str(IMAGE_LR),
    '--seed', str(SEED),
]

run(
    train_cmd,
    cwd=AACLIP_ROOT,
    env={'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'},
)


In [ ]:
print('Checkpoint files:')
for path in sorted(SAVE_PATH.glob('*.pth')):
    print(path.name, f'{path.stat().st_size / (1024 * 1024):.2f} MB')

log_path = SAVE_PATH / 'train.log'
if log_path.exists():
    print('\nLast train.log lines:')
    print('\n'.join(log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-20:]))


In [ ]:
RUN_EVAL_AFTER_TRAIN = False

if RUN_EVAL_AFTER_TRAIN:
    eval_cmd = [
        sys.executable, 'test.py',
        '--dataset', DATASET_NAME,
        '--shot', str(SHOT),
        '--save_path', str(SAVE_PATH),
        '--img_size', str(IMG_SIZE),
        '--batch_size', '16',
    ]
    run(eval_cmd, cwd=AACLIP_ROOT)
else:
    print('Skipping evaluation. Set RUN_EVAL_AFTER_TRAIN = True to run official test.py.')


In [ ]:
archive_path = shutil.make_archive(str(SAVE_PATH), 'zip', SAVE_PATH)
print('Zipped checkpoint artifact:', archive_path)
